In [4]:
import os
import sys
sys.dont_write_bytecode = True

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

# api keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# model selection
GENERATOR_MODEL_NAME = "openai/gpt-5"

# concurency level
RPS = 5
BATCH_SIZE = 10

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.data import (
    DatasetDeclaration,
    RAGDatasetAsyncGenerator,
    RerankingSample,
    CompressionSample
)
from core.messaging import (
    LangchainMessageBuilder,
    PROMPT_REGISTRY
)

# instntiate the generator
declarative_generator = RAGDatasetAsyncGenerator(
    # setting up client, prompts and parsers
    client=ChatOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=BASE_URL,
        model=GENERATOR_MODEL_NAME,
        max_tokens=4092,
        temperature=0.0,
        rate_limiter=InMemoryRateLimiter(
            requests_per_second=RPS,
            check_every_n_seconds=0.1,
            max_bucket_size=RPS * 2
        )
    ),
    # configuring dataset structure (domains, task-types, amount of examples per batch)
    declaration=DatasetDeclaration(
        tasks=["reranking", "context_compression"],
        domains=["coding", "history", "math", "medicine", "research"],
        difficulties=["easy", "medium", "complex"],
        batch_size=10
    ),
    # instantiating message builder for handling structured output
    messages_builder=LangchainMessageBuilder.from_sequence(
        ("reranking", PROMPT_REGISTRY.reranking_data_generation, RerankingSample),
        ("context_compression", PROMPT_REGISTRY.context_compression_data_generation, CompressionSample),
    ),
)

In [ ]:
rows = await declarative_generator.agenerate_dataset(output_dir="./tmp")